# PWV01 : Explore hologram data quality on PWV

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS
- creation date : 2025-09-16 (v1), adapted to the 2026 merged files on 2026-11-13
- last update : 2026-11-14 (data until July 2026)
- kernel : conda_py313 (python 3.13), package `mysitcom` installed in editable mode
- input : merged Spectractor files of `../2026-11-13-SpectractorExtraction-FromButler` (selected in `PWV00_parameters.py`)

**Goal** : show the night variations of PWV with date and time for the filters selected in `PWV_FILTER_LIST` (empty and OG550), fit a straight line night by night on the filters listed in `FIT_FILTERS` (drift in mm/h) and summarize the quality of each night.

Main changes with respect to the 2025 version :
- columns `PWV [mm]_x/_y` replaced by `PWV [mm]_rum` / `PWV [mm]_ram` (method chosen by `PWV_METHOD` in `PWV00_parameters.py`);
- `rum` and `ram` PWV are compared before selection;
- the cut on D_CCD is shown on the header value `D2CCD`, the one actually used in `getSelectionCut`;
- per-night aggregates are vectorized (`groupby`), the straight line fit is a plain weighted least squares;
- figures are saved in PNG and PDF, the per-night summary is saved in a csv file.

In [ ]:
import os
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
import seaborn as sns

warnings.simplefilter('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

plt.rcParams.update({
    'figure.figsize': (16, 8),
    'axes.labelsize': 'xx-large',
    'axes.titlesize': 'xx-large',
    'xtick.labelsize': 'xx-large',
    'ytick.labelsize': 'xx-large',
    'legend.fontsize': 'xx-large',
})
props = dict(boxstyle='round', facecolor='white', alpha=0.1)

print(f'python {python_version()}, numpy {np.__version__}, pandas {pd.__version__}')

## Configuration

In [ ]:
# everything related to the input file and to the cuts is defined in PWV00_parameters.py
from PWV00_parameters import (
    version_run, atmfilename, tag,
    FLAG_PWVFILTERS, PWV_FILTER_LIST,
    PWV_METHOD, PWV_COL, PWV_ERR_COL, OZONE_COL,
    CHI2_COL, D2CCD_COL, EXPTIME_COL, TARGET_COL, FILTER_COL, TIME_COL,
    DCCDMINFIG, DCCDMAXFIG, DCCDMINCUT, DCCDMAXCUT, CHI2CUT, EXPTIMECUT,
    FLAG_CORRECTFOR_PWV_REPEAT, FLAG_CORRECTFOR_PWV_REPEAT_RATIO, SIGMA_PWV_REPEAT, FACTORERR_PWV_REPEAT,
    site_lsst, get_astronomical_midnight, getSelectionCut, DumpConfig,
)
DumpConfig()

In [ ]:
# where the figures and the data products are stored
NB_TAG = 'PWV01'
prefix = f'pwv01_{PWV_METHOD}'
pathfigs = f'figs_{NB_TAG}'
pathdata = f'data_{NB_TAG}'
for path in (pathfigs, pathdata):
    os.makedirs(path, exist_ok=True)


def save_fig(fig, name, formats=('png', 'pdf')):
    '''Save a figure in the figure directory, with the notebook prefix.'''
    for ext in formats:
        fig.savefig(f'{pathfigs}/{prefix}_{name}.{ext}', bbox_inches='tight')

In [ ]:
PWVMIN, PWVMAX = 0., 20.          # y-range of the PWV plots [mm]

# the collimator was installed on 2023-09-30 : if True keep only the nights after,
# if False keep all nights and shade the period before the collimator
FLAG_WITHCOLLIMATOR = False
DATE_WITHCOLLIMATOR = 20230930
datetime_WITHCOLLIMATOR = pd.to_datetime('2023-09-30 00:00:00+0000')

PLOT_EACH_NIGHT = True            # make one figure per night (saved in PNG)
N_NIGHTS_DISPLAYED = 3            # number of per-night figures displayed in the notebook (all are saved)
FIT_FILTERS = ['empty']           # the straight line is fitted on the points taken with these filters
MIN_OBS_FIT = 11                  # minimum number of points (in FIT_FILTERS) in a night to fit a straight line

FILTER_COLORS = {'empty': 'blue', 
                 'BG40_65mm_1': 'green', 
                 'BG40': 'green', 
                 'OG550_65mm_1': 'red',
                 'OG550': 'red',
                 'FELH0600': 'purple'
                }

## Read the merged file
`atmfilename` is defined in `PWV00_parameters.py`.

In [ ]:
df_spec = pd.DataFrame(np.load(atmfilename, allow_pickle=True))

# time for plotting, night of observation and sequence number from the exposure id (YYYYMMDD + 5 digits)
df_spec['Time'] = pd.to_datetime(df_spec[TIME_COL], utc=True)
df_spec['nightObs'] = df_spec['id'] // 100_000
df_spec['seq_num'] = df_spec['id'] % 100_000

print(f'{len(df_spec)} spectra, nights {df_spec.nightObs.min()} -> {df_spec.nightObs.max()} ({df_spec.nightObs.nunique()} nights)')
df_spec.groupby(FILTER_COL).size()

### Check filters

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
sns.scatterplot(data=df_spec, x='Time', y='seq_num', hue=FILTER_COL, palette=FILTER_COLORS,
                s=60, edgecolor='black', linewidth=0.2, ax=ax)
ax.set_title(f'Auxtel Holo observations wrt date and filter type, {tag}')
ax.set_xlabel('Date of observation')
ax.set_ylabel('Seq Num')
ax.tick_params(axis='x', rotation=45)
ax.legend(bbox_to_anchor=(1.01, 1.0), ncols=1)
plt.tight_layout()
save_fig(fig, 'filters_vs_time')
plt.show()

## Select the filters

In [ ]:
if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec[FILTER_COL].isin(PWV_FILTER_LIST)].copy()

# nights with the collimator
if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec['nightObs'] > DATE_WITHCOLLIMATOR].copy()

print(f'{len(df_spec)} spectra kept, filters : {df_spec[FILTER_COL].unique()}')

### Visualize the selection cuts
The cut on the distance to the CCD is applied on the header value `D2CCD`.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 4))
ax1, ax2, ax3 = axs
df_spec.hist(D2CCD_COL, ax=ax1, bins=50, range=(DCCDMINFIG, DCCDMAXFIG), facecolor='b')
ax1.axvline(DCCDMINCUT, ls='-.', c='k')
ax1.axvline(DCCDMAXCUT, ls='-.', c='k')
df_spec.hist(CHI2_COL, ax=ax2, bins=50, range=(0, 300), facecolor='b')
ax2.axvline(CHI2CUT, ls='-.', c='k')
df_spec.hist(EXPTIME_COL, ax=ax3, bins=20, facecolor='b')
ax3.axvline(EXPTIMECUT, ls='-.', c='k')
plt.suptitle(f'Quality selection cuts, {tag}')
plt.tight_layout()
save_fig(fig, 'cuts_histograms')
plt.show()

In [ ]:
date_form = DateFormatter('%y-%m-%d')
fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
ax1, ax2 = axs

for filt, group in df_spec.groupby(FILTER_COL):
    ax1.scatter(group['Time'], group[D2CCD_COL], marker='+', lw=3, alpha=0.5, color=FILTER_COLORS.get(filt, 'k'), label=filt)
ax1.legend(ncols=3)
ax1.set_ylim(DCCDMINFIG, DCCDMAXFIG)
ax1.axhline(DCCDMINCUT, ls='-.', c='k')
ax1.axhline(DCCDMAXCUT, ls='-.', c='k')
ax1.set_ylabel(D2CCD_COL + ' [mm]')
ax1.set_title('Distance to CCD vs time')
ax1.grid()

for filt, group in df_spec.groupby(FILTER_COL):
    ax2.scatter(group['Time'], group[CHI2_COL], marker='+', lw=3, alpha=0.5, color=FILTER_COLORS.get(filt, 'k'), label=filt)
ax2.set_yscale('log')
ax2.axhline(CHI2CUT, ls='-.', c='k')
ax2.set_ylabel(CHI2_COL)
ax2.set_xlabel('time')
ax2.set_title('CHI2_FIT vs time')
ax2.grid()
ax2.xaxis.set_major_formatter(date_form)
ax2.tick_params(axis='x', rotation=45)

plt.suptitle(tag)
plt.tight_layout()
save_fig(fig, 'cuts_vs_time')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for filt, group in df_spec.groupby(FILTER_COL):
    ax.scatter(group['AIRMASS'], group[CHI2_COL], color=FILTER_COLORS.get(filt, 'k'), alpha=0.4, label=filt)
ax.legend()
ax.set_xlabel('Airmass')
ax.set_ylabel(CHI2_COL)
ax.set_yscale('log')
ax.axhline(CHI2CUT, ls='-.', c='k')
ax.grid(True)
plt.tight_layout()
save_fig(fig, 'chi2_vs_airmass')
plt.show()

### Atmospheric parameters before selection : `rum` (1D spectrum) versus `ram` (2D spectrogram)

In [ ]:
def clipped_range(series, qlow=0.01, qhigh=0.99):
    return series.quantile(qlow), series.quantile(qhigh)


fig, axs = plt.subplots(2, 3, figsize=(20, 10))
ax1, ax2, ax3, ax4, ax5, ax6 = axs.flatten()
for ax, method, color in ((ax1, 'rum', 'b'), (ax2, 'ram', 'g')):
    col = f'PWV [mm]_{method}'
    df_spec.hist(col, ax=ax, bins=50, range=clipped_range(df_spec[col]), facecolor=color)
for ax, method, color in ((ax4, 'rum', 'b'), (ax5, 'ram', 'g')):
    col = f'ozone [db]_{method}'
    df_spec.hist(col, ax=ax, bins=50, range=clipped_range(df_spec[col]), facecolor='r')

ax3.scatter(df_spec['PWV [mm]_rum'], df_spec['PWV [mm]_ram'], s=4, alpha=0.3)
ax3.plot([0, PWVMAX], [0, PWVMAX], 'k--')
ax3.set_xlim(0, PWVMAX)
ax3.set_ylim(0, PWVMAX)
ax3.set_xlabel('PWV rum [mm]')
ax3.set_ylabel('PWV ram [mm]')
ax3.set_title('ram vs rum')

ax6.scatter(df_spec['ozone [db]_rum'], df_spec['ozone [db]_ram'], s=4, alpha=0.3, color='r')
ax6.set_xlabel('ozone rum [db]')
ax6.set_ylabel('ozone ram [db]')
ax6.set_title('ram vs rum')

plt.suptitle(f'Atmospheric parameters before selection, {tag}')
plt.tight_layout()
save_fig(fig, 'atmparams_before_selection')
plt.show()

### Targets used

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
sns.stripplot(data=df_spec, x='Time', y=TARGET_COL, hue=TARGET_COL, palette='Set1',
              size=6, jitter=True, edgecolor='black', linewidth=0.1, legend=False, ax=ax)
ax.set_title(f'Auxtel Holo observations wrt date and target, {tag}')
ax.set_xlabel('Time')
ax.set_ylabel('Target')
ax.tick_params(axis='x', rotation=45)
ax.grid()
plt.tight_layout()
save_fig(fig, 'targets_vs_time')
plt.show()

In [ ]:
# faint stars (the first four of the list of faint calspec stars)
LIST_OF_FAINT_TARGETS = ['Feige110', 'HD074000', 'HD115169', 'HD031128']
df_spec['isFaint'] = df_spec[TARGET_COL].isin(LIST_OF_FAINT_TARGETS)
df_spec.groupby(TARGET_COL)['isFaint'].agg(['size', 'first'])

## Errors on PWV : include or not the repeatability
The corrected error is stored in a new column, so that re-running the cell does not apply the correction twice.

In [ ]:
ERR_COL = 'PWV_err_used'
if FLAG_CORRECTFOR_PWV_REPEAT:
    if FLAG_CORRECTFOR_PWV_REPEAT_RATIO:
        df_spec[ERR_COL] = df_spec[PWV_ERR_COL] * FACTORERR_PWV_REPEAT
    else:
        df_spec[ERR_COL] = np.sqrt(df_spec[PWV_ERR_COL]**2 + SIGMA_PWV_REPEAT**2)
else:
    df_spec[ERR_COL] = df_spec[PWV_ERR_COL]

## Per-night statistics (before the quality selection)

In [ ]:
def nightly_stats(df):
    '''Per-night statistics of the PWV and of the CHI2 of the fit.'''
    g = df.groupby('nightObs')
    return pd.DataFrame({
        'count': g[PWV_COL].count(),
        'pwv_mean': g[PWV_COL].mean(),
        'pwv_median': g[PWV_COL].median(),
        'pwv_std': g[PWV_COL].std(),
        'chi2_mean': g[CHI2_COL].mean(),
        'chi2_median': g[CHI2_COL].median(),
        'chi2_std': g[CHI2_COL].std(),
    })


def plot_night_summary(stats, label, name):
    '''Bar plots per night of the number of spectra, median and std of PWV, median CHI2.'''
    items = (('count', 'Number of measurements per night', 'b'),
             ('pwv_median', 'Median PWV per night [mm]', 'b'),
             ('pwv_std', 'STD of PWV per night [mm]', 'b'),
             ('chi2_median', 'Median CHI2 per night', 'r'))
    fig, axs = plt.subplots(len(items), 1, figsize=(18, 4 * len(items)))
    for ax, (col, title, color) in zip(axs, items):
        stats[col].plot(kind='bar', ax=ax, color=color, grid=True, rot=90)
        ax.set_title(f'{title}, {label}, {tag}', fontsize=16)
    plt.tight_layout()
    save_fig(fig, name)
    plt.show()
    return fig


stats_all = nightly_stats(df_spec)
_ = plot_night_summary(stats_all, 'before selection', 'nightly_stats_before_selection')

## Night boundaries and astronomical midnights

In [ ]:
def get_night_boundaries(df, margin=pd.Timedelta(minutes=30)):
    '''Dictionary nightObs -> (first observation - margin, last observation + margin).'''
    b = df.groupby('nightObs')['Time'].agg(['min', 'max'])
    return {night: (row['min'] - margin, row['max'] + margin) for night, row in b.iterrows()}


def get_night_midnights(nights):
    '''Dictionary nightObs -> astronomical midnight (UTC timestamp). Takes about 0.3 s per night.'''
    d = {}
    for night in nights:
        day = datetime.strptime(str(night), '%Y%m%d').date()
        ts = pd.Timestamp(get_astronomical_midnight(site_lsst, day))
        d[night] = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    return d


dn = get_night_boundaries(df_spec)
dnidnights = get_night_midnights(sorted(df_spec['nightObs'].unique()))

## Plot all data

In [ ]:
def plot_pwv_vs_time(df, title, name, bounds, midnights):
    '''PWV versus time for all the points, with the night boundaries and the midnights.'''
    fig, ax = plt.subplots(figsize=(18, 8))
    for filt, group in df.groupby(FILTER_COL):
        ax.errorbar(group['Time'], group[PWV_COL], yerr=group[ERR_COL], fmt='o', label=filt,
                    color=FILTER_COLORS.get(filt, 'k'), ecolor='k', capsize=1, markersize=5)
    for tmin, tmax in bounds.values():
        ax.axvspan(tmin, tmax, color='blue', alpha=0.05)
    for midnight in midnights.values():
        ax.axvline(midnight, color='purple', ls=':', alpha=0.5)
    if not FLAG_WITHCOLLIMATOR:
        ax.axvspan(df['Time'].min(), datetime_WITHCOLLIMATOR, color='yellow', alpha=0.1)
    ax.set_ylabel(PWV_COL)
    ax.set_xlabel('Time')
    ax.xaxis.set_major_formatter(DateFormatter('%y-%m-%d'))
    ax.tick_params(axis='x', rotation=45)
    ax.set_title(f'{title}, {tag}')
    ax.set_ylim(PWVMIN, PWVMAX)
    ax.legend(loc='upper right')
    ax.grid()
    plt.tight_layout()
    save_fig(fig, name)
    plt.show()


plot_pwv_vs_time(df_spec, 'Precipitable water vapor measured by holo vs time (before selection cuts)',
                 'pwv_allpoints_allnights_noqualcuts', dn, dnidnights)

## Apply the quality selection cuts

In [ ]:
cut = getSelectionCut(df_spec)
df_spec_sel = df_spec[cut].reset_index(drop=True)
print(f'{len(df_spec_sel)} / {len(df_spec)} spectra kept after the quality cuts ({100 * len(df_spec_sel) / len(df_spec):.1f} %)')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for ax, method, color in zip(axs, ('rum', 'ram'), ('b', 'g')):
    col = f'PWV [mm]_{method}'
    df_spec_sel.hist(col, ax=ax, bins=50, range=(PWVMIN, PWVMAX), facecolor=color)
plt.suptitle(f'PWV after selection, {tag}')
plt.tight_layout()
save_fig(fig, 'pwv_after_selection')
plt.show()

In [ ]:
dn = get_night_boundaries(df_spec_sel)
plot_pwv_vs_time(df_spec_sel, 'Precipitable water vapor measured by holo vs time (after selection cuts)',
                 'pwv_allpoints_allnights_withqualcuts', dn, dnidnights)

In [ ]:
stats_sel = nightly_stats(df_spec_sel)
_ = plot_night_summary(stats_sel, 'after selection', 'nightly_stats_after_selection')

## Time variation night by night
For each night : statistics of the PWV (all filters) and weighted straight line fit `PWV = a + b * dt` on the points of `FIT_FILTERS`, `dt` in hours since the first fitted observation of the night, slope `b` in mm/h.

In [ ]:
def fit_line(x, y, yerr):
    '''Weighted least squares fit of y = a + b x. Returns (a, b), their errors, and chi2/ndof.
    The covariance matrix is rescaled by chi2/ndof.'''
    w = 1. / yerr**2
    A = np.vstack([np.ones_like(x), x]).T
    cov = np.linalg.inv(A.T @ (A * w[:, None]))
    params = cov @ (A.T @ (w * y))
    chi2dof = np.sum(((y - A @ params) / yerr)**2) / (len(x) - 2)
    cov *= chi2dof
    return params, np.sqrt(np.diag(cov)), chi2dof


def plot_night(d, night, t0, params, stats, display):
    '''PWV and airmass versus time for one night, with the fitted line.'''
    fig, ax = plt.subplots(figsize=(18, 8))
    ax_bis = ax.twinx()
    for filt, group in d.groupby(FILTER_COL):
        color = FILTER_COLORS.get(filt, 'k')
        ax.errorbar(group['Time'], group[PWV_COL], yerr=group[ERR_COL], fmt='o', label=filt,
                    color=color, ecolor='k', capsize=5, markersize=10)
        ax_bis.scatter(group['Time'], group['AIRMASS'], color=color, marker='+')
    if params is not None:
        xfit = np.linspace(0., (d['Time'].max() - t0).total_seconds() / 3600., 50)
        ax.plot(t0 + pd.to_timedelta(xfit, unit='h'), params[0] + params[1] * xfit, 'k-')
    ax.axvline(dnidnights[night], color='r', ls='--', lw=2)

    ax_bis.set_ylim(2.5, 0.5)
    ax_bis.set_ylabel('airmass (+)')
    ax.set_ylabel(PWV_COL)
    ax.set_xlabel('time (UTC)')
    ax.set_ylim(PWVMIN, PWVMAX)
    ax.xaxis.set_major_formatter(DateFormatter('%y-%m-%dT%H:%M'))
    ax.tick_params(axis='x', labelrotation=45)
    ax.grid()
    ax.legend(loc='upper right')
    text1 = '\n'.join((f"count : {stats['count']} (fit : {stats['count_fit']})", f"mean : {stats['mean']:.1f} mm",
                       f"median : {stats['median']:.1f} mm", f"std : {stats['std']:.1f} mm"))
    text2 = '\n'.join(('d(PWV)/dt :', f"slope : {stats['slope']:.3f} mm/h",
                       f"slope err : {stats['slope_err']:.3f} mm/h", f"chi2/ndof : {stats['chi2dof']:.2f}"))
    ax.text(0.03, 0.95, text1, transform=ax.transAxes, fontsize=14, va='top', bbox=props)
    ax.text(0.30, 0.95, text2, transform=ax.transAxes, fontsize=14, va='top', bbox=props)
    ax.set_title(f'night {night}, {tag}')
    plt.tight_layout()
    save_fig(fig, f'per_night_{night}', formats=('png',))
    if display:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
rows = []
n_displayed = 0
for night, d in df_spec_sel.groupby('nightObs'):
    d = d.sort_values('Time')
    pwv = d[PWV_COL]
    stats = {'nightObs': night, 'count': len(d), 'mean': pwv.mean(), 'median': pwv.median(), 'std': pwv.std(),
             'count_fit': 0, 'slope': np.nan, 'slope_err': np.nan, 'chi2dof': np.nan}
    params = None
    d_fit = d[d[FILTER_COL].isin(FIT_FILTERS)]
    stats['count_fit'] = len(d_fit)
    if len(d_fit) >= MIN_OBS_FIT:
        t0 = d_fit['Time'].min()
        x = ((d_fit['Time'] - t0).dt.total_seconds() / 3600.).values
        try:
            params, perr, chi2dof = fit_line(x, d_fit[PWV_COL].values, d_fit[ERR_COL].values)
            stats.update(slope=params[1], slope_err=perr[1], chi2dof=chi2dof)
        except np.linalg.LinAlgError:
            params = None
        if PLOT_EACH_NIGHT:
            plot_night(d, night, t0, params, stats, display=(n_displayed < N_NIGHTS_DISPLAYED))
            n_displayed += 1
    rows.append(stats)

df_night = pd.DataFrame(rows).set_index('nightObs')
df_night['Time'] = pd.to_datetime(df_night.index.astype(str), format='%Y%m%d')
df_night.to_csv(f'{pathdata}/{prefix}_nightly_summary_{version_run}.csv')
print(f'{len(df_night)} nights, {df_night.slope.notna().sum()} with a fitted drift ({FIT_FILTERS})')
df_night.head()

## Summary of the night quality

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
ax1, ax2 = axs

ax1.errorbar(df_night['Time'], df_night['median'], yerr=df_night['std'], fmt='o', ms=5, color='r', ecolor='k')
ax1.set_ylabel('PWV (mm)')
ax1.set_title(f'Median Precipitable water vapor per night, {tag}')
ax1.set_ylim(PWVMIN, PWVMAX)
ax1.grid()

ax2.errorbar(df_night['Time'], df_night['slope'], yerr=df_night['slope_err'], fmt='.', color='r', ecolor='k', ms=10)
ax2.set_title('Fitted drift per night')
ax2.set_ylabel('dPWV/dt (mm/h)')
ax2.set_xlabel('time')
ax2.set_ylim(-1., 1.)
ax2.grid()
ax2.xaxis.set_major_formatter(DateFormatter('%y-%m-%d'))
ax2.tick_params(axis='x', labelrotation=45)

plt.tight_layout()
save_fig(fig, 'pwv_medians_and_slopes_allnights')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
ax.errorbar(df_night['Time'], df_night['median'], yerr=df_night['std'], fmt='o', ms=5, color='r', ecolor='k')
ax.set_ylabel('PWV (mm)')
ax.set_xlabel('time')
ax.xaxis.set_major_formatter(DateFormatter('%y-%m-%d'))
ax.set_title(f'Median and spread of Precipitable water vapor per night, {tag}')
ax.set_ylim(PWVMIN, PWVMAX)
ax.grid()
if not FLAG_WITHCOLLIMATOR:
    ax.axvspan(df_night['Time'].min(), datetime_WITHCOLLIMATOR.tz_localize(None), color='yellow', alpha=0.1)
plt.tight_layout()
save_fig(fig, 'pwv_medians_allnights')
plt.show()